# 04 — Gemma4 via Ollama (Kaggle GPU, local) — retrieval gate

Local-GPU arm for Gemma. Installs the Ollama binary, starts `ollama serve`, pulls **`gemma4:e4b`** (9.6GB, 128K ctx, Text+Image in — biggest Gemma4 that fits a T4 16GB). No fallback. Model weights live under `/tmp/ollama_models` (`OLLAMA_MODELS`), not the working dir.

Same fairness contract as 02: SAME document, SAME question (`+ " Output the answer directly."`), `temperature=0 / num_predict=128`, only the context representation changes. Token accounting here uses Ollama's own counters: `prompt_eval_count` per arm (decoder-visible input length as reported by the runtime) plus `eval_count` (generated). Report `CR_decoder_ollama = prompt_eval_text / prompt_eval_image`, labeled as such — it is NOT the same instrument as the HF-processor `m/k` in 02, so never mix the two in one column without labeling. VRAM: the Ollama server owns the GPU, so per-call `torch.cuda` stats are meaningless here — we sample `nvidia-smi` used-memory per row instead (expect ~constant: model stays resident).

In [ ]:
# Cell 0 — fetch repo (branch Sayok) on Kaggle; safe to re-run
import os, subprocess
if os.path.isdir("/kaggle/working"):
    dest = "/kaggle/working/Testing-dip"
    if os.path.isdir(os.path.join(dest, ".git")):
        subprocess.run(["git", "-C", dest, "pull", "--ff-only"], check=False)
    elif not os.path.isdir(dest):
        subprocess.run(["git", "clone", "-b", "Sayok",
                        "https://github.com/iamahnaf/Testing-dip.git", dest], check=True)
    print("repo:", dest)
else:
    print("not on Kaggle — skipping repo fetch (run from the repo checkout)")


In [ ]:
# Cell 1 — config (no fallback)
OLLAMA_MODEL = "gemma4:e4b"   # 9.6GB, fits T4 16GB — no fallback
BUDGETS = [{"tag": "low", "canvas_w": 600}, {"tag": "high", "canvas_w": 1100}]
CONTEXTS = [{"tag": "short", "words": 350}, {"tag": "med", "words": 1500}, {"tag": "long", "words": 3000}]
GEN = {"temperature": 0.0, "num_predict": 128}
OUT = "/kaggle/working/results/raw/gemma4_ollama.jsonl"
print(OLLAMA_MODEL, BUDGETS, GEN)

In [ ]:
# Cell 2 — install Ollama, start server, pull model (slow: ~10GB download, run once)
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -2
!mkdir -p /tmp/ollama_models && export OLLAMA_MODELS=/tmp/ollama_models
!pkill ollama 2>/dev/null; sleep 1; OLLAMA_MODELS=/tmp/ollama_models nohup ollama serve > /tmp/ollama.log 2>&1 & sleep 4; OLLAMA_MODELS=/tmp/ollama_models ollama list
!OLLAMA_MODELS=/tmp/ollama_models ollama pull $OLLAMA_MODEL 2>&1 | tail -3  # weights to /tmp, not working dir
print("server log tail:", open("/tmp/ollama.log").read()[-400:])

In [ ]:
# Cell 3 — renderer + samples + metrics + Ollama helpers (same logic as 01/02)
import os
os.environ["OLLAMA_MODELS"] = "/tmp/ollama_models"
import base64, io, json, random, re, string, subprocess, textwrap, time, unicodedata, urllib.request
from PIL import Image, ImageDraw, ImageFont

def normalize(t):
    t = unicodedata.normalize("NFKC", t or "")
    return re.sub(r"[ \t]+", " ", t.replace("\r\n", "\n")).strip()

def render(text, width=800, font_size=18, ls=6, wrap=80, pad=30):
    text = normalize(text)
    try: font = ImageFont.truetype("arial.ttf", font_size)
    except IOError: font = ImageFont.load_default()
    lines = []
    for p in text.split("\n\n"):
        lines.extend(textwrap.wrap(p, width=wrap)); lines.append("")
    img = Image.new("RGB", (width, 2*pad + max(len(lines)*(font_size+ls), 100)), "white")
    d = ImageDraw.Draw(img); y = pad
    for ln in lines:
        if ln: d.text((pad, y), ln, fill="black", font=font)
        y += font_size + ls
    return img

DIST = ("The thermal radiator on the secondary boom radiates excess heat into cold space. "
        "Star tracking sensor arrays refresh coordinate quaternions at thirty hertz. ")
def make_sample(words, seed):
    rng = random.Random(seed)
    code = f"CIPHER-{rng.randint(1000, 9999)}-OMEGA"
    needle = f"Special authorization key {code} was recorded in emergency protocol ledger section 9."
    paras, n = [], 0
    while n < words:
        p = " ".join(rng.choice([DIST]) for _ in range(6)); paras.append(p); n += len(p.split())
    w = paras[len(paras)//2].split(); w.insert(len(w)//2, needle)
    paras[len(paras)//2] = " ".join(w)
    return "\n\n".join(paras), code

def norm(s):
    return re.sub(r"\s+", " ", (s or "").lower().translate(str.maketrans("", "", string.punctuation))).strip()
def tok_f1(pred, gt):
    p, g = set(norm(pred).split()), set(norm(gt).split())
    if not p or not g: return 0.0
    tp = len(p & g); prec, rec = tp/len(p), tp/len(g)
    return 0.0 if prec+rec == 0 else 2*prec*rec/(prec+rec)

def img_b64(img):
    buf = io.BytesIO(); img.save(buf, "PNG")
    return base64.b64encode(buf.getvalue()).decode()

def gpu_used_mb():
    try:
        out = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], text=True)
        return float(out.strip().splitlines()[0])
    except Exception:
        return None

def ollama_chat(content, images=None, retries=3):
    """POST /api/chat, stream=false. Returns (text, counters dict). Times end-to-end locally."""
    msg = {"role": "user", "content": content}
    if images: msg["images"] = images
    body = json.dumps({"model": OLLAMA_MODEL, "messages": [msg], "stream": False,
        "options": {"temperature": GEN["temperature"], "num_predict": GEN["num_predict"]}}).encode()
    for a in range(retries):
        try:
            s = time.perf_counter()
            req = urllib.request.Request("http://localhost:11434/api/chat", data=body, headers={"Content-Type": "application/json"})
            r = json.load(urllib.request.urlopen(req, timeout=600))
            lat = time.perf_counter() - s
            return (r.get("message", {}).get("content", ""), {
                "prompt_eval_count": r.get("prompt_eval_count"), "eval_count": r.get("eval_count"),
                "total_duration_s": round(r.get("total_duration", 0)/1e9, 2),
                "load_duration_s": round(r.get("load_duration", 0)/1e9, 2),
                "end_to_end_s": round(lat, 2)})
        except Exception as e:
            if a < retries-1: time.sleep(10*(a+1)); continue
            raise RuntimeError(f"ollama chat failed: {e} — is `ollama serve` up? see /tmp/ollama.log")

# Warm-up (excludes model load from steady-state timing) + reachability
t0, c0 = ollama_chat("Reply with exactly: OK")
print("warm-up:", t0.strip()[:50], c0, "gpu_used_mb:", gpu_used_mb())

In [ ]:
# Cell 4 — A/B loop: text arm vs image arm, Ollama counters, JSONL log (Sec 21)
from datetime import datetime, timezone
os.makedirs(os.path.dirname(OUT), exist_ok=True)
QSUFFIX = " Output the answer directly."
n = 0
for ci, c in enumerate(CONTEXTS):
    context, gt = make_sample(c["words"], seed=100+ci)  # SAME seeds as 02/03 → same documents
    question = "What special authorization key was recorded in emergency protocol ledger section 9?" + QSUFFIX
    pred_t, ct = ollama_chat(f"Context:\n{context}\n\nQuestion: {question}")
    gpu_t = gpu_used_mb()
    for b in BUDGETS:
        img = render(context, width=b["canvas_w"])
        pred_i, ci_ = ollama_chat(f"Question: {question}", images=[img_b64(img)])
        gpu_i = gpu_used_mb()
        pt, pi = ct["prompt_eval_count"], ci_["prompt_eval_count"]
        rec = {"experiment_id": "gemma4_ollama", "timestamp": datetime.now(timezone.utc).isoformat(),
               "model": OLLAMA_MODEL, "runtime": "ollama-serve-local", "task": "ruler-sniah", "input_mode": "paired",
               "context_tag": c["tag"], "visual_budget_tag": b["tag"], "canvas_w": b["canvas_w"],
               "image_size": f"{img.width}x{img.height}", "renderer": "pil-fallback",
               "prompt_eval_text": pt, "prompt_eval_image": pi,
               "CR_decoder_ollama": round(pt/pi, 3) if (pt and pi) else None,
               "eval_count_text": ct["eval_count"], "eval_count_image": ci_["eval_count"],
               "hit_text": int(norm(gt) in norm(pred_t)), "hit_image": int(norm(gt) in norm(pred_i)),
               "f1_text": round(tok_f1(pred_t, gt), 3), "f1_image": round(tok_f1(pred_i, gt), 3),
               "end_to_end_text_s": ct["end_to_end_s"], "end_to_end_image_s": ci_["end_to_end_s"],
               "total_duration_text_s": ct["total_duration_s"], "total_duration_image_s": ci_["total_duration_s"],
               "gpu_used_text_mb": gpu_t, "gpu_used_image_mb": gpu_i,
               "temperature": GEN["temperature"], "num_predict": GEN["num_predict"],
               "ground_truth": gt, "pred_text": pred_t.strip()[:300], "pred_image": pred_i.strip()[:300]}
        open(OUT, "a").write(json.dumps(rec) + "\n"); n += 1
        print(f"[{n}/6] {c['tag']} x {b['tag']}: prompt_eval T/I={pt}/{pi} CRdec={rec['CR_decoder_ollama']} hitT/I={rec['hit_text']}/{rec['hit_image']} tT/I={ct['end_to_end_s']}/{ci_['end_to_end_s']}s")
print(f"\nLOGGED {n} paired rows -> {OUT}")

In [ ]:
# Cell 5 — aggregate from the JSONL (never hand-edit)
import pandas as pd
df = pd.read_json(OUT, lines=True)
g = df.groupby("visual_budget_tag").agg(n=("hit_image", "size"), acc_text=("hit_text", "mean"),
    acc_image=("hit_image", "mean"), f1_image=("f1_image", "mean"),
    cr_dec=("CR_decoder_ollama", "mean"), t_text=("end_to_end_text_s", "mean"), t_image=("end_to_end_image_s", "mean"))
print(g.round(3).to_string())
print("\nGATE: pass iff all 6 rows logged and hit/f1 non-null. NOTE: CR_decoder_ollama uses Ollama runtime counters — compare against 02/03 CR columns only with the instrument labeled (see notebook header).")